In [0]:
# Crear volumen para checkpoints en Unity Catalog
spark.sql("CREATE VOLUME IF NOT EXISTS proyecto_bi.silver.checkpoints")

print("✅ Volumen creado correctamente")

In [0]:
# Verificar la ruta del volumen
checkpoint_path = "/Volumes/proyecto_bi/silver/checkpoints/streaming_nyctaxi"
print(f"Ruta checkpoint: {checkpoint_path}")

In [0]:
# Leer la tabla Silver como fuente de streaming
# Esto simula que los datos llegan en tiempo real en micro-lotes
df_stream = (
    spark.readStream
    .format("delta")
    .option("maxFilesPerTrigger", 1)  # Simula 1 archivo por micro-lote
    .table("proyecto_bi.silver.yellow_trips")
)

print("✅ Stream configurado correctamente")
print(f"Schema del stream:")
df_stream.printSchema()

In [0]:
from pyspark.sql.functions import col, current_timestamp

checkpoint_path = "/Volumes/proyecto_bi/silver/checkpoints/streaming_nyctaxi"

df_stream_procesado = df_stream.select(
    col("tpep_pickup_datetime"),
    col("PULocationID").alias("zona"),
    col("total_amount"),
    col("trip_distance"),
    col("passenger_count"),
    current_timestamp().alias("processed_at")
)

query = (
    df_stream_procesado
    .writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .outputMode("append")
    .trigger(processingTime="10 seconds")
    .toTable("proyecto_bi.gold.streaming_viajes")
)

print("✅ Stream iniciado — procesando micro-lotes cada 10 segundos")

In [0]:
# Ver el estado del stream en tiempo real
display(query.lastProgress)

In [0]:
%sql
SELECT COUNT(*) AS total_procesados 
FROM proyecto_bi.gold.streaming_viajes;

In [0]:
%sql
SELECT
    DATE(processed_at)            AS fecha_proceso,
    COUNT(*)                      AS viajes_procesados,
    ROUND(SUM(total_amount), 2)   AS ingreso_total
FROM proyecto_bi.gold.streaming_viajes
GROUP BY DATE(processed_at)
ORDER BY fecha_proceso DESC
LIMIT 10;

In [0]:
query.stop()
print("✅ Stream detenido correctamente")